In [484]:
import kagglehub
import pandas as pd

In [485]:
path = kagglehub.dataset_download('jasminemohamed2545/berber-english-160k-parallel-sentences-for-nlp')
print(path)

C:\Users\lassa\.cache\kagglehub\datasets\jasminemohamed2545\berber-english-160k-parallel-sentences-for-nlp\versions\1


# Preparing & Restructuring Data

1. load & read data, check dtypes
2. rename columns
3. remove unnecessary column
4. Create `tifinagh` column 

In [486]:
df = pd.read_table('ber.txt')
df.head()

,Go.,Ṛuḥ.,CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #8325200 (Yagurten)
0,Hi.,Azul.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
1,Hi.,Azul fell-ak.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
2,Hi.,Azul fell-am.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
3,Hi.,Azul fell-awen.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Azul fell-awent.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...


In [487]:
df.dtypes

Go.                                                                                object
Ṛuḥ.                                                                               object
CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #8325200 (Yagurten)    object
dtype: object

In [488]:
# renaming columns
df.columns = ['english', 'amazigh', 'license']
df.head()

,english,amazigh,license
0,Hi.,Azul.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
1,Hi.,Azul fell-ak.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
2,Hi.,Azul fell-am.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
3,Hi.,Azul fell-awen.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Azul fell-awent.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...


In [489]:
# removing license column and saving changes to df_clean dataframe
df_clean = df.drop(columns=['license'])
df_clean.head()

,english,amazigh
0,Hi.,Azul.
1,Hi.,Azul fell-ak.
2,Hi.,Azul fell-am.
3,Hi.,Azul fell-awen.
4,Hi.,Azul fell-awent.


In [490]:
# creating a Tifinagh script column
df_clean['tifinagh'] = pd.NA
df_clean.head(10)

,english,amazigh,tifinagh
0,Hi.,Azul.,<NA>
1,Hi.,Azul fell-ak.,<NA>
2,Hi.,Azul fell-am.,<NA>
3,Hi.,Azul fell-awen.,<NA>
4,Hi.,Azul fell-awent.,<NA>
5,Hi.,ⴰⵣⵓⵍ.,<NA>
6,Hi.,azul,<NA>
7,Run!,Azzel!,<NA>
8,Run!,ⴰⵣⵣⵍ,<NA>
9,Run.,Azzel.,<NA>


# Filtering

1. Detect tifinagh unicode characters
2. Separate Latin and Tifinagh entries
3. inspect/fix the mixed Latin–Tifinagh row
4. Create `script` column


In [491]:
# detect tifinagh characters: unicode U+2D30–U+2D7F
tifinagh_mask = df_clean['amazigh'].str.contains(r'[\u2D30-\u2D7F]',regex=True,na=False)
tifinagh_mask[:10]

0    False
1    False
2    False
3    False
4    False
5     True
6    False
7    False
8     True
9    False
Name: amazigh, dtype: bool

In [492]:
# copy tifinagh entries
df_clean.loc[tifinagh_mask, 'tifinagh'] = df_clean.loc[tifinagh_mask, 'amazigh']

# remove tifinagh entries from amazigh
df_clean.loc[tifinagh_mask, 'amazigh'] = pd.NA

# classify writing system
df_clean['script'] = 'Latin'
df_clean.loc[tifinagh_mask, 'script'] = 'tifinagh'

df_clean[:15]

,english,amazigh,tifinagh,script
0,Hi.,Azul.,<NA>,Latin
1,Hi.,Azul fell-ak.,<NA>,Latin
2,Hi.,Azul fell-am.,<NA>,Latin
3,Hi.,Azul fell-awen.,<NA>,Latin
4,Hi.,Azul fell-awent.,<NA>,Latin
5,Hi.,<NA>,ⴰⵣⵓⵍ.,tifinagh
6,Hi.,azul,<NA>,Latin
7,Run!,Azzel!,<NA>,Latin
8,Run!,<NA>,ⴰⵣⵣⵍ,tifinagh
9,Run.,Azzel.,<NA>,Latin


In [493]:
# check whether the amazigh output is in Latin or Tifinagh 
df_clean['script'].value_counts()

script
Latin       160825
tifinagh        57
Name: count, dtype: int64

In [ ]:
# remove missing values (NaN, None) with .dropna()
# this will allow to check all tifinagh entries available
df_clean[['english', 'tifinagh']].dropna()

,english,tifinagh
5,Hi.,ⴰⵣⵓⵍ.
8,Run!,ⴰⵣⵣⵍ
13,Wow!,ⵜⵉⵍⵉⵍⴰ
14,Wow!,ⵜⵉⵍⵉⵍⴰ !
20,Fire!,ⵜⴰⴽⴰⵜ
22,Help!,ⴰⵡⵙ
23,Help!,ⴰⵡⵙ !
24,Jump.,ⵏⴹⴻⵔ
25,Jump.,ⵏⴹⴻⵕ
28,Stop!,ⴱⵉⴷⴷ


In [495]:
df_clean.loc[135529]

english              He doesn't know anything about politics.
amazigh                                                  <NA>
tifinagh    Ur issin awed ḥaḥ g tsertit. ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ...
script                                               tifinagh
Name: 135529, dtype: object

In [496]:
df_clean.loc[135529, 'amazigh'] = 'Ur issin awed ḥaḥ g tsertit'
df_clean.loc[135529, 'tifinagh'] = 'ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ⴳ ⵜⵙⴻⵔⵜⵉⵜ'

In [497]:
df_clean.loc[135529]

english     He doesn't know anything about politics.
amazigh                  Ur issin awed ḥaḥ g tsertit
tifinagh                  ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ⴳ ⵜⵙⴻⵔⵜⵉⵜ
script                                      tifinagh
Name: 135529, dtype: object

row 135529 has a mix of tifinagh and latin concatenated together -> will separate the two

In [498]:
df_clean[['english', 'amazigh', 'tifinagh']].dropna()

,english,amazigh,tifinagh
135529,He doesn't know anything about politics.,Ur issin awed ḥaḥ g tsertit,ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ⴳ ⵜⵙⴻⵔⵜⵉⵜ


In [499]:
df_clean[['english', 'tifinagh']].dropna().head()

,english,tifinagh
5,Hi.,ⴰⵣⵓⵍ.
8,Run!,ⴰⵣⵣⵍ
13,Wow!,ⵜⵉⵍⵉⵍⴰ
14,Wow!,ⵜⵉⵍⵉⵍⴰ !
20,Fire!,ⵜⴰⴽⴰⵜ


## Distribution Findings
- Majority of amazigh data is written in Latin, Tifinagh is scarce 
- Solution: either manually add Tifinagh equivalent or focus only on Latin version

# Text Cleaning

1. Punctuation removal
2. Whitespace stripping
3. Check for missing values
4. Check for duplicates
5. Check for Inconsistent Text & Typos 

In [500]:
# remove punctuation 
df_clean['english'] = df_clean['english'].str.replace(r'[^\w\s]', '', regex=True)
df_clean['amazigh'] = df_clean['amazigh'].str.replace(r'[^\w\s]', '', regex=True)
df_clean['tifinagh'] = df_clean['tifinagh'].str.replace(r'[^\w\s]', '', regex=True)

# remove extra spaces
df_clean['english'] = df_clean['english'].str.strip()
df_clean['amazigh'] = df_clean['amazigh'].str.strip()
df_clean['tifinagh'] = df_clean['tifinagh'].str.strip()

In [501]:
df_clean.loc[20:25]
# space and punctuation successfully removed

,english,amazigh,tifinagh,script
20,Fire,<NA>,ⵜⴰⴽⴰⵜ,tifinagh
21,Help,Abbuh,<NA>,Latin
22,Help,<NA>,ⴰⵡⵙ,tifinagh
23,Help,<NA>,ⴰⵡⵙ,tifinagh
24,Jump,<NA>,ⵏⴹⴻⵔ,tifinagh
25,Jump,<NA>,ⵏⴹⴻⵕ,tifinagh


## Missing Data

In [502]:
# identifying missing data
df_clean[df_clean.isna().any(axis=1)]

,english,amazigh,tifinagh,script
0,Hi,Azul,<NA>,Latin
1,Hi,Azul fellak,<NA>,Latin
2,Hi,Azul fellam,<NA>,Latin
3,Hi,Azul fellawen,<NA>,Latin
4,Hi,Azul fellawent,<NA>,Latin
...,...,...,...,...
160877,One of my favorite quotes by Mark Twain is Its...,Yiwet seg tnebdurininu timenyafin n Mark Twain...,<NA>,Latin
160878,If you talk to a man in a language he understa...,Mer ad astemmeslayeḍ i umdan s tutlayt i ifehh...,<NA>,Latin
160879,You cant view Flash content on an iPad However...,Ur tezmired ara ad twalid agbur n Flash deg us...,<NA>,Latin
160880,You cant view Flash content on an iPad However...,Ur tezmired ara ad twalid agbur n Flash deg us...,<NA>,Latin


In [503]:
df_clean.isna().sum() 

english          0
amazigh         56
tifinagh    160825
script           0
dtype: int64

In [504]:
df_clean.dtypes

english     object
amazigh     object
tifinagh    object
script      object
dtype: object

In [505]:
# converting datatypes 'object' to 'string'
# df_clean['english'] = df_clean['english'].astype('string')
# df_clean['amazigh'] = df_clean['amazigh'].astype('string')
# df_clean['tifinagh'] = df_clean['tifinagh'].astype('string')

# df_clean['script'] = df_clean['script'].astype('string')

## Duplicate Data

In [506]:
# checking for duplicate translation pairs
df_clean.duplicated().sum()

np.int64(183)

In [507]:
df_clean[df_clean.duplicated(keep=False)]

,english,amazigh,tifinagh,script
7,Run,Azzel,<NA>,Latin
9,Run,Azzel,<NA>,Latin
13,Wow,<NA>,ⵜⵉⵍⵉⵍⴰ,tifinagh
14,Wow,<NA>,ⵜⵉⵍⵉⵍⴰ,tifinagh
22,Help,<NA>,ⴰⵡⵙ,tifinagh
...,...,...,...,...
143649,Tom went to the library just to see Mary,Tom iṛuḥ ɣer temkarḍit akken kan ad iẓer Mary,<NA>,Latin
155548,If Tom had a lot of money hed buy that for you,Lemmer yesɛi Tom aṭas n yidrimen tili ad awent...,<NA>,Latin
155549,If Tom had a lot of money hed buy that for you,Lemmer yesɛi Tom aṭas n yidrimen tili ad awent...,<NA>,Latin
155554,If Tom had a lot of money hed buy that for you,Lemmer ili Tom aṭas n yidrimen tili ad awentti...,<NA>,Latin


In [508]:
# dropping duplicates & resetting index
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.reset_index(drop=True)

In [509]:
df_clean.duplicated().sum()

np.int64(0)

In [510]:
df_clean['english'].duplicated().sum()

np.int64(76601)

In [511]:
df_clean[df_clean['english'].duplicated(keep=False)].sort_values('english')
# duplicates appear because of gender markers e.g., nne-k and nne-m suffixes in amazigh

,english,amazigh,tifinagh,script
87543,100 years is called a century,Meyya n yiseggasen ttininas lqern,<NA>,Latin
87542,100 years is called a century,Tawinest n yiseggasen ttininas lqern,<NA>,Latin
93674,A bad habit is easily acquired,Yir tanamit fessuset i ulmad,<NA>,Latin
93673,A bad habit is easily acquired,Fessus ad yelmed yiwen yir tanamit,<NA>,Latin
153865,A bicycle will rust if you leave it in the rain,Lemmer ad teǧǧed tasnasɣalt i unẓar ad tebṛec,<NA>,Latin
...,...,...,...,...
2095,Youve won,Tellummẓeḍ,<NA>,Latin
2094,Youve won,Tellummẓed,<NA>,Latin
2093,Youve won,Trebḥemt,<NA>,Latin
2090,Youve won,Trebḥed,<NA>,Latin


## Inconsistent Text & Typos Check

The data has a mix of Tifinagh script with Latin which is inconsistent, I will either try to convert all the Tifinagh into Latin, Latin to Tifinagh, or include both in separate columns. 

In [512]:
df_clean.describe()

,english,amazigh,tifinagh,script
count,160699,160648,52,160699
unique,84098,149215,50,2
top,Go home now,Ur yettett ara waya,ⴰⵣⵓⵍ,Latin
freq,56,14,2,160647


# Translation Variations

In [513]:
(df_clean.groupby('english')["amazigh"]
 .nunique()
 .sort_values(ascending=False)
 .head(20))


english
Go home now                                 56
They hated you                              48
Do as you want                              45
Choose whichever you like                   42
You have lots of friends                    41
I want you to succeed                       40
They want you back                          40
You still have enough time                  38
Do you think that this can work             37
Do you think this can work                  37
You arent the only one Tom has cheated      36
Dont think Im going to let you do that      34
Have you finished writing the letter yet    33
They want to see you                        32
They might be taller than you               32
Do you really want to win                   32
They are pleased with your work             32
Youre not the only one who has a camera     32
Youre prepared                              32
Do they like you                            32
Name: amazigh, dtype: int64

### Translation Variation Findings

The corpus shows a high degree of translation variation, with many English sentences corresponding to multiple unique Amazigh translations. For example, “Go home now” has 56 unique Amazigh translations in the dataset.

Rather than treating these observations as duplicates, they may represent meaningful linguistic variation within the corpus.

Possible sources of variation include:

- Gender marking in Amazigh
- Singular and plural inflection
- Regional and dialectal variation across Amazigh varieties
- Lexical variation and alternative vocabulary choices

These patterns will be investigated further before additional cleaning or normalization decisions are made.


In [514]:
# df_clean[df_clean["english"] == "Go home now"][["english", "amazigh"]]

# Exporting the Cleaned Dataset

In [515]:
df_clean.to_csv('amazigh_english_clean.csv', index=False)
df_clean.to_pickle('amazigh_english_clean.pkl')